In [4]:
import sys 
from pathlib import Path

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data" 

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import pandas as pd

In [5]:
df=pd.read_csv(DATA_DIR / 'processed_data.csv')

In [6]:
df

,subject,exercise,time index,acc_x_u1,acc_x_u2,acc_x_u3,acc_x_u4,acc_x_u5,acc_y_u1,acc_y_u2,...,gyr_mag_u3,mag_mag_u3,acc_mag_u4,gyr_mag_u4,mag_mag_u4,acc_mag_u5,gyr_mag_u5,mag_mag_u5,most_active_unit,label
0,s1,e1,1,-9.685645,-9.268303,-3.490739,9.018139,-4.191311,-1.645149,2.981149,...,0.026881,0.822163,9.757600,0.014221,0.787898,9.813711,0.008580,0.762010,u2,correct
1,s1,e1,2,-9.648184,-9.260987,-3.445827,9.044963,-4.188912,-1.645353,2.958806,...,0.024231,0.821653,9.772105,0.034212,0.786162,9.793846,0.016693,0.761298,u2,correct
2,s1,e1,3,-9.700570,-9.260972,-3.475073,9.077281,-4.181551,-1.615223,2.966184,...,0.022021,0.822319,9.843089,0.033648,0.786961,9.813023,0.019365,0.761755,u2,correct
3,s1,e1,4,-9.685627,-9.260754,-3.474895,9.032722,-4.181585,-1.630183,3.010663,...,0.028818,0.820716,9.772934,0.018946,0.786664,9.798967,0.013698,0.762076,u2,correct
4,s1,e1,5,-9.655697,-9.260783,-3.467499,8.993878,-4.186467,-1.630194,2.995905,...,0.030181,0.820915,9.749011,0.031250,0.787620,9.797769,0.024000,0.761082,u2,correct
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126381,s5,e8,5088,-1.527953,-9.674701,1.978098,6.922647,1.876655,-9.665978,1.265019,...,0.025282,0.899167,9.785526,0.023613,0.931566,9.813756,0.016301,0.932956,u2,low_amplitude
126382,s5,e8,5089,-1.527883,-9.689552,1.962998,6.927831,1.879036,-9.628649,1.265227,...,0.029466,0.899541,9.799846,0.006585,0.933682,9.722012,0.018409,0.932929,u2,low_amplitude
126383,s5,e8,5090,-1.542963,-9.734492,1.992746,6.922948,1.859529,-9.665803,1.228453,...,0.015660,0.899638,9.787181,0.018174,0.932079,9.736470,0.024049,0.932213,u2,low_amplitude
126384,s5,e8,5091,-1.542924,-9.689592,1.948238,6.898431,1.854705,-9.650913,1.272519,...,0.019069,0.898545,9.742179,0.027622,0.933872,9.773205,0.008863,0.933159,u2,low_amplitude


In [8]:
inputs=[
        
        'acc_x_u1', 'acc_x_u2', 'acc_x_u3', 'acc_x_u4', 'acc_x_u5', 
        'acc_y_u1', 'acc_y_u2', 'acc_y_u3', 'acc_y_u4', 'acc_y_u5', 
        'acc_z_u1', 'acc_z_u2', 'acc_z_u3', 'acc_z_u4', 'acc_z_u5',
        'gyr_x_u1', 'gyr_x_u2', 'gyr_x_u3', 'gyr_x_u4', 'gyr_x_u5', 
        'gyr_y_u1', 'gyr_y_u2', 'gyr_y_u3', 'gyr_y_u4', 'gyr_y_u5',
        'gyr_z_u1', 'gyr_z_u2', 'gyr_z_u3', 'gyr_z_u4', 'gyr_z_u5', 
        'mag_x_u1', 'mag_x_u2', 'mag_x_u3', 'mag_x_u4', 'mag_x_u5', 
        'mag_y_u1', 'mag_y_u2', 'mag_y_u3', 'mag_y_u4', 'mag_y_u5', 
        'mag_z_u1', 'mag_z_u2', 'mag_z_u3', 'mag_z_u4', 'mag_z_u5',
]

In [ ]:
exercises = ['e1','e2','e3','e4','e5','e6','e7','e8']
subject=['s1', 's2', 's3', 's5']
labels=['correct', 'low_amplitude', 'fast']

feature_rows = []

for s in subject:
    for e in exercises:
        for l in labels:

            df_r = df[(df['subject'] == s) &
                      (df['exercise'] == e) &
                      (df['label'] == l)]

            if len(df_r) == 0:
                continue

            duration = df_r['time index'].max() - df_r['time index'].min()
            sample_duration = duration / 10

            t0 = df_r['time index'].min()

            for rep in range(10):

                start = t0 + rep * sample_duration
                end = t0 + (rep + 1) * sample_duration

                df_rep = df_r[(df_r['time index'] >= start) &
                              (df_r['time index'] < end)]

                if len(df_rep) == 0:
                    continue

                row = {
                    "subject": s,
                    "exercise": e,
                    "label": l,
                    "rep": rep
                }

                for col in inputs:
                    row[col + "_mean"] = df_rep[col].mean()
                    row[col + "_std"] = df_rep[col].std()

                feature_rows.append(row)

    

In [12]:
feature_df = pd.DataFrame(feature_rows)

print(feature_df.shape)
feature_df.head(15)

(960, 94)


,subject,exercise,label,rep,acc_x_u1_mean,acc_x_u1_std,acc_x_u2_mean,acc_x_u2_std,acc_x_u3_mean,acc_x_u3_std,...,mag_z_u1_mean,mag_z_u1_std,mag_z_u2_mean,mag_z_u2_std,mag_z_u3_mean,mag_z_u3_std,mag_z_u4_mean,mag_z_u4_std,mag_z_u5_mean,mag_z_u5_std
0,s1,e1,correct,0,-9.668909,0.016958,-3.902571,4.049221,-2.771116,0.546597,...,-0.069455,0.004053,-0.107993,0.046744,-0.143677,0.050158,-0.589262,0.007369,0.232330,0.001595
1,s1,e1,correct,1,-9.666975,0.018536,-3.039581,3.998590,-2.612466,0.507640,...,-0.063115,0.002620,-0.116241,0.040746,-0.166548,0.032331,-0.594483,0.004363,0.230809,0.003557
2,s1,e1,correct,2,-9.668994,0.017615,-4.140788,4.406814,-2.677855,0.490694,...,-0.059568,0.002364,-0.095974,0.053675,-0.154062,0.038826,-0.593862,0.004348,0.233980,0.003084
3,s1,e1,correct,3,-9.669289,0.017997,-2.662014,3.829392,-2.546875,0.461191,...,-0.057733,0.001190,-0.121011,0.038925,-0.168551,0.032274,-0.597984,0.004528,0.232466,0.003324
4,s1,e1,correct,4,-9.673880,0.017801,-3.738403,4.411501,-2.626196,0.464645,...,-0.055841,0.001793,-0.104693,0.059519,-0.158296,0.042133,-0.597444,0.002921,0.233950,0.005232
5,s1,e1,correct,5,-9.675472,0.018751,-3.535743,4.313884,-2.618399,0.434220,...,-0.055744,0.001287,-0.107227,0.060303,-0.160311,0.041862,-0.598068,0.003968,0.234023,0.003225
6,s1,e1,correct,6,-9.675224,0.018634,-3.698027,4.289189,-2.644203,0.478492,...,-0.054589,0.001327,-0.108929,0.050343,-0.159799,0.036710,-0.598682,0.003263,0.234657,0.003767
7,s1,e1,correct,7,-9.676189,0.016260,-3.359147,4.056068,-2.615628,0.437213,...,-0.054346,0.001411,-0.123257,0.045327,-0.170736,0.033100,-0.599567,0.004928,0.231227,0.003481
8,s1,e1,correct,8,-9.669731,0.017373,-3.085235,4.118134,-2.592612,0.450916,...,-0.053028,0.001981,-0.124705,0.047002,-0.172600,0.034873,-0.599799,0.003391,0.229392,0.003803
9,s1,e1,correct,9,-9.668297,0.020543,-2.997856,4.258542,-2.565810,0.485288,...,-0.056720,0.006437,-0.115166,0.055012,-0.164437,0.044688,-0.596930,0.012066,0.229428,0.002552
